# 50 — Phase 0 retrieval diagnostic + offline-eval workflow

**Two purposes**:
1. Produce the **bare-retriever baseline** (BM25 / dense / fused wRRF) on the full dev split — drives Phase 1 bundle prioritization.
2. Establish the **offline-evaluation workflow** so future experiments (BGE-M3, Qwen3-Embedding-4B, doc2query, contrastive FT) can be compared against this baseline with paired-bootstrap statistical significance — no Blind-A budget burned on uncertain bets.

**Why offline-first?** Blind-A has 80 rows with ±0.05 nDCG@20 noise per submission and a ~3/week cap. The dev split has ~8000 turns — paired bootstrap on it gives a far tighter signal. The rule we'll enforce: only submit to Blind-A when the **dev paired-bootstrap CI excludes 0** with a positive mean delta.

**Hardware**: A100-40GB or RTX PRO 6000 Blackwell (95 GB). Runtime ~10–15 min on A100 (full dev, ~8000 turns, batch_size=64).

**What this notebook ships**:
- `scripts/phase0_retrieval_diagnostic.py` — runs the diagnostic, emits summary JSON + markdown + per-turn records JSONL
- `scripts/compare_diagnostic_runs.py` — compares two records JSONLs with paired-bootstrap CIs, failure-mode migration matrix, per-slice deltas
- 31 unit tests across both (TDD'd)


In [ ]:
# 1) GPU check.
!nvidia-smi | head -10

In [ ]:
# 2) Clone fresh-model branch.
BRANCH = 'fresh-model'
!rm -rf /content/recsys2026
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026
%cd /content/recsys2026

In [ ]:
# 3) HF auth + Drive cache.
import os
from google.colab import userdata
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN set from Colab secrets.')
except Exception as e:
    print('NO HF_TOKEN — set it in Colab secrets (left sidebar > key icon) before running cell 5.', e)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'
    print('HF_HOME =', os.environ['HF_HOME'])
except Exception as e:
    print('Drive not mounted (using ephemeral Colab cache).', e)

In [ ]:
# 4) Install deps (retrieval-side only; no training stack).
!pip install -q --upgrade transformers datasets 'pandas<3.0' tqdm omegaconf pyyaml bm25s scipy numpy

In [ ]:
# 5) Optional smoke (5 sessions, ~1 min) — catches wiring issues before the full run.
!python scripts/phase0_retrieval_diagnostic.py \
    --sample 5 --batch-size 16 \
    --out-json /tmp/phase0_smoke.json \
    --out-md /tmp/phase0_smoke.md \
    --out-records /tmp/phase0_smoke_records.jsonl
print('--- smoke summary ---')
print(open('/tmp/phase0_smoke.md').read())

In [ ]:
# 6) Full dev run. Defaults: config/110-prorank-rerank-devset.yaml, full dev (~8000 turns),
#    writes summary JSON + markdown + per-turn records JSONL.
!python scripts/phase0_retrieval_diagnostic.py \
    --batch-size 64 \
    --out-json data/phase0_diagnostic.json \
    --out-md documents/phase0_baseline_report.md \
    --out-records data/phase0_diagnostic_records.jsonl

In [ ]:
# 7) Render baseline report inline + headline summary.
from IPython.display import Markdown, display
display(Markdown(open('documents/phase0_baseline_report.md').read()))
print('\n--- headline numbers ---')
import json
summary = json.load(open('data/phase0_diagnostic.json'))
print('n_turns:', summary['n_turns'])
for comp, m in summary['per_component_metrics'].items():
    print(f"  {comp:8s}  recall@20={m['recall@20']:.3f}  recall@100={m['recall@100']:.3f}  ndcg@20={m['ndcg@20']:.3f}")
print('failure_breakdown:', summary['failure_breakdown'])

## Offline-eval workflow — comparing future experiments to this baseline

**The pattern for every Phase 1+ experiment**:

1. Create / use a config YAML that swaps the component you want to test (e.g., a BGE-M3 retriever or a Qwen3-Embedding-4B retriever).
2. Re-run the diagnostic with that config and a *different* output path:
   ```bash
   python scripts/phase0_retrieval_diagnostic.py \
       --config music-crs-baselines/config/<experiment>.yaml \
       --out-records data/<experiment>_records.jsonl \
       --out-md documents/<experiment>_report.md \
       --out-json data/<experiment>_diagnostic.json
   ```
3. Compare with the comparison utility:
   ```bash
   python scripts/compare_diagnostic_runs.py \
       --baseline data/phase0_diagnostic_records.jsonl \
       --experiment data/<experiment>_records.jsonl \
       --label-baseline 'BM25 + Qwen-0.6B (current)' \
       --label-experiment '<experiment label>' \
       --out-md documents/<experiment>_vs_baseline.md
   ```

**The headline output**: `FUSED Δ recall@20 = +X.XXXX [low, high] — verdict`. If the verdict is `improved (CI excludes 0)` with a meaningful mean delta (≥ +0.02 to be worth a Blind-A slot), ship it. If `no significant change`, the experiment didn't move the needle even at 8000 turns of statistical power — don't waste a Blind submission.

**Decision criteria**:
| Verdict | Action |
|---|---|
| `improved (CI excludes 0)` AND mean Δ ≥ +0.02 | Submit to Blind-A; record on submissions log |
| `improved (CI excludes 0)` AND mean Δ < +0.02 | Keep in ensemble (per ensemble-first principle); don't burn Blind-A slot solo |
| `no significant change` | Pivot to a different bundle |
| `regressed (CI excludes 0)` | Don't ship; document the negative result |


In [ ]:
# 8) Template — run an experiment with a different config (uncomment and edit when ready).
#
# Example: Phase 1 Bundle B (Qwen3-Embedding-4B). Requires creating
# music-crs-baselines/config/bge_m3_retrieval.yaml first (or whatever experiment).
#
# !python scripts/phase0_retrieval_diagnostic.py \
#     --config music-crs-baselines/config/bge_m3_retrieval.yaml \
#     --batch-size 64 \
#     --out-json data/bge_m3_diagnostic.json \
#     --out-md documents/bge_m3_report.md \
#     --out-records data/bge_m3_records.jsonl

In [ ]:
# 9) Template — compare experiment to baseline (uncomment when both records files exist).
#
# !python scripts/compare_diagnostic_runs.py \
#     --baseline data/phase0_diagnostic_records.jsonl \
#     --experiment data/bge_m3_records.jsonl \
#     --label-baseline 'BM25 + Qwen3-Embedding-0.6B + wRRF (current)' \
#     --label-experiment 'BGE-M3' \
#     --out-md documents/bge_m3_vs_baseline.md \
#     --out-json data/bge_m3_vs_baseline.json
#
# from IPython.display import Markdown, display
# display(Markdown(open('documents/bge_m3_vs_baseline.md').read()))

In [ ]:
# 10) Optional — commit baseline artifacts back to the branch (so they're anchored in git).
# Records JSONL can be ~50-100MB; if that's too large for git, commit only the summary + markdown
# and stash records on Drive instead.
!git config user.email 'orrimoch@gmail.com'
!git config user.name 'Or Rimoch (Colab)'
!git add data/phase0_diagnostic.json documents/phase0_baseline_report.md
# !git add data/phase0_diagnostic_records.jsonl   # uncomment if size is OK
!git commit -m 'phase0: bare-retriever diagnostic baseline on full dev'
# !git push origin {BRANCH}   # uncomment when ready to push